***操作步骤***

***1 创建项目目录***

mkdir ~/mcp_auth_service
cd ~/mcp_auth_service

***2 创建文件 mcp_server.py，内容如下：***

In [ ]:
#!/usr/bin/env python3
import sys
import json
import secrets
import time

# ---------- 配置 ----------
# 是否使用固定 token（方便调试）；若为 None 则每次启动随机生成
FIXED_TOKEN = "my_secret_token_2026"   # 可改为 None 使用随机 token

# 全局 token
if FIXED_TOKEN:
    TOKEN = FIXED_TOKEN
else:
    TOKEN = secrets.token_hex(16)

# ---------- MCP 协议处理 ----------
def send_response(response):
    """向 stdout 发送 JSON-RPC 响应"""
    print(json.dumps(response), flush=True)

def handle_initialize(id):
    """处理 initialize 请求"""
    return {
        "jsonrpc": "2.0",
        "id": id,
        "result": {
            "protocolVersion": "0.1.0",
            "capabilities": {
                "tools": {}   # 支持工具调用
            },
            "serverInfo": {
                "name": "auth-mcp-server",
                "version": "1.0.0"
            }
        }
    }

def handle_tools_list(id):
    """返回可用工具列表"""
    tools = [
        {
            "name": "get_token",
            "description": "获取访问受保护工具所需的 token（无需鉴权）",
            "inputSchema": {
                "type": "object",
                "properties": {},
                "required": []
            }
        },
        {
            "name": "get_server_time",
            "description": "返回服务器当前时间（需要提供 token 鉴权）",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "token": {
                        "type": "string",
                        "description": "通过 get_token 工具获取的 token"
                    }
                },
                "required": ["token"]
            }
        }
    ]
    return {
        "jsonrpc": "2.0",
        "id": id,
        "result": {
            "tools": tools
        }
    }

def handle_tools_call(id, params):
    """执行工具调用"""
    tool_name = params.get("name")
    arguments = params.get("arguments", {})

    # 工具2：get_token（无需鉴权）
    if tool_name == "get_token":
        return {
            "jsonrpc": "2.0",
            "id": id,
            "result": {
                "content": [
                    {
                        "type": "text",
                        "text": f"Token: {TOKEN}\n\n请妥善保管此 token，在调用 get_server_time 时作为参数传入。"
                    }
                ]
            }
        }

    # 工具1：get_server_time（需要鉴权）
    if tool_name == "get_server_time":
        provided_token = arguments.get("token")
        if not provided_token:
            return {
                "jsonrpc": "2.0",
                "id": id,
                "error": {
                    "code": -32000,
                    "message": "鉴权失败：缺少 token 参数"
                }
            }
        if provided_token != TOKEN:
            return {
                "jsonrpc": "2.0",
                "id": id,
                "error": {
                    "code": -32001,
                    "message": "鉴权失败：token 无效"
                }
            }

        # 鉴权通过，返回当前时间（功能自定）
        current_time = time.strftime("%Y-%m-%d %H:%M:%S")
        return {
            "jsonrpc": "2.0",
            "id": id,
            "result": {
                "content": [
                    {
                        "type": "text",
                        "text": f"服务器当前时间：{current_time}"
                    }
                ]
            }
        }

    # 未知工具
    return {
        "jsonrpc": "2.0",
        "id": id,
        "error": {
            "code": -32601,
            "message": f"方法未找到：{tool_name}"
        }
    }

def main():
    """主循环：从 stdin 读取 JSON-RPC 请求并响应"""
    while True:
        line = sys.stdin.readline()
        if not line:
            break
        line = line.strip()
        if not line:
            continue

        try:
            req = json.loads(line)
        except json.JSONDecodeError:
            # 忽略无效 JSON
            continue

        req_id = req.get("id")
        method = req.get("method")

        if method == "initialize":
            resp = handle_initialize(req_id)
        elif method == "tools/list":
            resp = handle_tools_list(req_id)
        elif method == "tools/call":
            resp = handle_tools_call(req_id, req.get("params", {}))
        else:
            # 不支持的方法
            resp = {
                "jsonrpc": "2.0",
                "id": req_id,
                "error": {
                    "code": -32601,
                    "message": f"方法未实现：{method}"
                }
            }
        send_response(resp)

if __name__ == "__main__":
    main()

***3 给此脚本设置权限：chmod +x mcp_server.py***

***4 创建测试客户端脚本：在同一目录下创建 test_client.py：***

cd ~/mcp_auth_service
nano test_client.py

In [ ]:
#!/usr/bin/env python3
import subprocess
import json
import time

# 启动 MCP 服务进程
proc = subprocess.Popen(
    ["python3", "mcp_server.py"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

def send_request(req):
    """发送 JSON-RPC 请求并读取响应"""
    proc.stdin.write(json.dumps(req) + "\n")
    proc.stdin.flush()
    line = proc.stdout.readline()
    return json.loads(line)

# 1. 初始化连接
init_req = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "0.1.0",
        "clientInfo": {"name": "test-client", "version": "1.0"}
    }
}
resp = send_request(init_req)
print("[1] Initialize response:", json.dumps(resp, indent=2))

# 2. 获取工具列表
tools_req = {"jsonrpc": "2.0", "id": 2, "method": "tools/list"}
resp = send_request(tools_req)
print("[2] Tools list:", json.dumps(resp, indent=2))

# 3. 调用无鉴权工具 get_token
get_token_req = {
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "get_token",
        "arguments": {}
    }
}
resp = send_request(get_token_req)
print("[3] get_token response:", json.dumps(resp, indent=2))

# 提取 token
token_text = resp["result"]["content"][0]["text"]
token = token_text.split("Token: ")[1].split("\n")[0]
print(f"    -> Extracted token: {token}")

# 4. 调用需要鉴权的工具（使用错误 token）
bad_req = {
    "jsonrpc": "2.0",
    "id": 4,
    "method": "tools/call",
    "params": {
        "name": "get_server_time",
        "arguments": {"token": "wrong_token"}
    }
}
resp = send_request(bad_req)
print("[4] With invalid token:", json.dumps(resp, indent=2))

# 5. 调用需要鉴权的工具（使用正确 token）
good_req = {
    "jsonrpc": "2.0",
    "id": 5,
    "method": "tools/call",
    "params": {
        "name": "get_server_time",
        "arguments": {"token": token}
    }
}
resp = send_request(good_req)
print("[5] With valid token:", json.dumps(resp, indent=2))

# 结束服务进程
proc.terminate()
time.sleep(0.5)
proc.kill()

***5 运行测试 ：python3 test_client.py得到输出***

In [ ]:
"""
(main) root@23a50b17298b:~/mcp_auth_service# python3 test_client.py
[1] Initialize response: {
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "protocolVersion": "0.1.0",
    "capabilities": {
      "tools": {}
    },
    "serverInfo": {
      "name": "auth-mcp-server",
      "version": "1.0.0"
    }
  }
}
[2] Tools list: {
  "jsonrpc": "2.0",
  "id": 2,
  "result": {
    "tools": [
      {
        "name": "get_token",
        "description": "\u83b7\u53d6\u8bbf\u95ee\u53d7\u4fdd\u62a4\u5de5\u5177\u6240\u9700\u7684 token\uff08\u65e0\u9700\u9274\u6743\uff09",
        "inputSchema": {
          "type": "object",
          "properties": {},
          "required": []
        }
      },
      {
        "name": "get_server_time",
        "description": "\u8fd4\u56de\u670d\u52a1\u5668\u5f53\u524d\u65f6\u95f4\uff08\u9700\u8981\u63d0\u4f9b token \u9274\u6743\uff09",
        "inputSchema": {
          "type": "object",
          "properties": {
            "token": {
              "type": "string",
              "description": "\u901a\u8fc7 get_token \u5de5\u5177\u83b7\u53d6\u7684 token"
            }
          },
          "required": [
            "token"
          ]
        }
      }
    ]
  }
}
[3] get_token response: {
  "jsonrpc": "2.0",
  "id": 3,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Token: my_secret_token_2026\n\n\u8bf7\u59a5\u5584\u4fdd\u7ba1\u6b64 token\uff0c\u5728\u8c03\u7528 get_server_time \u65f6\u4f5c\u4e3a\u53c2\u6570\u4f20\u5165\u3002"
      }
    ]
  }
}
    -> Extracted token: my_secret_token_2026
[4] With invalid token: {
  "jsonrpc": "2.0",
  "id": 4,
  "error": {
    "code": -32001,
    "message": "\u9274\u6743\u5931\u8d25\uff1atoken \u65e0\u6548"
  }
}
[5] With valid token: {
  "jsonrpc": "2.0",
  "id": 5,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "\u670d\u52a1\u5668\u5f53\u524d\u65f6\u95f4\uff1a2026-05-29 00:22:21"
      }
    ]
  }
}
"""